In [1]:
from common import *

Function subs: 4
1st deriv subs: 16
2nd deriv subs: 64


# $DP$ sector

We find that $D_m P_n = A_{mn} + iB_{mn}$, where

$$A_{mn} = \frac{\nabla_m \tau_2 \nabla_n \tau_2}{2\tau_2^2} - \frac{\nabla_m \nabla_n \tau_2}{2\tau_2} - \frac{\nabla_m \tau_1 \nabla_n \tau_1}{2\tau_2^2} = -\tau_2 \, R_{1m1n} + \tfrac{1}{2} g^{ab} R_{ambn}$$

$$B_{mn} = \frac{\nabla_m \nabla_n \tau_1}{2\tau_2} - \frac{\nabla_m \tau_1 \nabla_n \tau_2}{2\tau_2^2} - \frac{\nabla_m \tau_2 \nabla_n \tau_1}{2\tau_2^2} = \tau_1 \, R_{1m1n} - \tfrac{1}{2}(R_{1m2n} + R_{2m1n})$$

and correspondingly $D_m \bar{P}_n = A_{mn} - iB_{mn}$.

---

Note that alternatively, one can define
$$v^a = (\tau,; -1)$$

Then:

$$\boxed{D_m P_n = \frac{1}{2\tau_2}, v^a v^b, R_{ambn}}, \qquad \boxed{D_m \bar{P}n = \frac{1}{2\tau_2}, \bar{v}^a \bar{v}^b, R{ambn}}$$

where $v^a$ is a null vector of the complexified torus metric: $g_{ab} v^a v^b = 0$, with normalization $g_{ab} v^a \bar{v}^b = 2\tau_2$.
The inverse metric decomposes as:
$$g^{ab} = \frac{v^a \bar{v}^b + \bar{v}^a v^b}{2\tau_2}$$

In [20]:
_A_mn = -tau2 * R_ambn[m,n][0,0] + sp.Rational(1,2) * sp.simplify(sum(ginv_func[a,b] * R_ambn[(m,n)][a,b] for a in range(2) for b in range(2)))
_B_mn = tau1 * R_ambn[m,n][0,0] - sp.Rational(1,2) * (R_ambn[m,n][0,1] + R_ambn[m,n][1,0])

sp.simplify(DP[m,n] - _A_mn - sp.I * _B_mn)

0

# $\text{Re} \bar{P}_m P_n$
Here we show that 
$$
\text{Re} \bar{P}_m P_n = \frac{1}{4\tau_2^2}(\nabla_m\tau_1 \nabla_n\tau_1 + \nabla_m\tau_2 \nabla_n\tau_2) = -\frac{1}{2} g^{ab} R_{ambn}
$$

In [37]:
PbP_mn = sp.expand(sp.simplify(real_parts(func2sym(Pb[m] * P[n]))))

trR_mn = - sp.Rational(1,2) * sum(ginv_func[a,b] * R_ambn[(m,n)][a,b] for a in range(2) for b in range(2))
trR_mn = sp.simplify(func2sym(trR_mn))

assert sp.simplify(PbP_mn - trR_mn) == 0

# $D_m P_n D_r \bar{P}_s$
Here we show that
$$
D_m P_n D_r \bar{P}_s = \frac{1}{4} (g^{ac} g^{bd} - \epsilon^{ac} \epsilon^{bd}) R_{ambn} R_{crds}
$$

In [22]:
DPDPb_mnrs = real_parts(DPb_with_R[m,n] * DP_with_R[r,s])

RR_mnrs = 0
for a,c in iterprod(range(2), range(2)):
    ginv_ac = func2sym(ginv_func[a,c])
    epsilon_ac = func2sym(epsilon2_up[a,c])
    for b,d in iterprod(range(2), range(2)):
        ginv_bd = func2sym(ginv_func[b,d])  
        epsilon_bd = func2sym(epsilon2_up[b,d])  
        Riemann_ambn =Rd_ambn_sym[a, m, b, n]
        Riemann_crds = Rd_ambn_sym[c, r, d, s]
        RR_mnrs += sp.Rational(1,4) * (ginv_ac * ginv_bd - epsilon_ac * epsilon_bd) * Riemann_ambn * Riemann_crds

sp.simplify(DPDPb_mnrs - RR_mnrs)

0

0

# $DG_3$ sector
$|DG_3|^2 = (\nabla F_4)_a (\nabla F_4)^a$

In [17]:
nablaF4_sq = nablaF4_down[m].T * (ginv_func * nablaF4_down[n])
nablaF4_sq = func2sym(nablaF4_sq)[0]
nablaF4_sq = sp.expand(sp.simplify(nablaF4_sq))

DG3DG3b = DG3b[m] * DG3[n]
DG3DG3b = real_parts(func2sym(DG3DG3b))
DG3DG3b = func2sym(DG3DG3b)
DG3DG3b = sp.expand(sp.simplify(DG3DG3b))

assert sp.simplify(nablaF4_sq - DG3DG3b) == 0

$\text{Re} D_m P_n D_r \bar{G}_3 D_s \bar{G}_3 = \frac{1}{2} (g^{ac} g^{bd} - \epsilon^{ac} \epsilon^{bd}) R_{ambn} \nabla_r F_{4c} \nabla_s F_{4d}$

In [18]:
RnablaF4nablaF4_mnrs = 0
for a,c in iterprod(range(2), range(2)):
    ginv_ac = func2sym(ginv_func[a,c])
    epsilon_ac = func2sym(epsilon2_up[a,c])
    for b,d in iterprod(range(2), range(2)):
        ginv_bd = func2sym(ginv_func[b,d])  
        epsilon_bd = func2sym(epsilon2_up[b,d])  
        Riemann_ambn = R_ambn[m,n][a,b]
        nabla_r_F4_c = nablaF4_down[r][c]
        nabla_s_F4_d = nablaF4_down[s][d]
        RnablaF4nablaF4_mnrs += sp.Rational(1,2) * (ginv_ac * ginv_bd - epsilon_ac * epsilon_bd) * func2sym(Riemann_ambn * nabla_r_F4_c * nabla_s_F4_d)

DPDG3bDG3b = real_parts(func2sym(DP[m,n] * DG3b[r] * DG3b[s]))

assert sp.simplify(RnablaF4nablaF4_mnrs - DPDG3bDG3b) == 0


0

0

In [20]:
DPb_expr = sp.expand(sp.simplify(func2sym(DPb[(m, n)])))
R_expr = sp.expand(sp.simplify(func2sym(R_ambn[(m, n)])))

tau1_s, tau2_s = SYM_D0[tau1], SYM_D0[tau2]

c00, c01, c10, c11 = sp.symbols('c00 c01 c10 c11')
diff = sp.expand(DPb_expr - (c00*R_expr[0,0] + c01*R_expr[0,1] + c10*R_expr[1,0] + c11*R_expr[1,1]))

# collect one equation per derivative symbol
deriv_syms = diff.free_symbols - {tau1_s, tau2_s, c00, c01, c10, c11}
eqs = [sp.Eq(diff.coeff(ds), 0) for ds in deriv_syms]
# constant term (no derivative symbols)
eqs.append(sp.Eq(diff.subs([(ds, 0) for ds in deriv_syms]), 0))

sol = sp.solve(eqs, [c00, c01, c10, c11])


In [21]:
sol

{c00: (\tau_{1}**2 - 2*I*\tau_{1}*\tau_{2} - \tau_{2}**2)/(2*\tau_{2}),
 c01: (-\tau_{1} + I*\tau_{2})/(2*\tau_{2}),
 c10: (-\tau_{1} + I*\tau_{2})/(2*\tau_{2}),
 c11: 1/(2*\tau_{2})}

In [15]:
DPb_expr = sp.expand(sp.simplify(func2sym(DPb[(m, n)])))
R_expr = sp.expand(sp.simplify(func2sym(R_ambn[(m, n)])))

# unknowns
c00, c01, c10, c11 = sp.symbols('c00 c01 c10 c11')

eq = sp.Eq(DPb_expr,
           c00*R_expr[0,0] + c01*R_expr[0,1] + c10*R_expr[1,0] + c11*R_expr[1,1])

sol = sp.solve(eq, [c00, c01, c10, c11])


In [17]:
DPb_expr

-I*\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) - \nabla_{m} \nabla_{n} \tau_{2}/(2*\tau_{2}) - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}/(2*\tau_{2}**2) + I*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}/(2*\tau_{2}**2) + I*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{1}/(2*\tau_{2}**2) + \nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}/(2*\tau_{2}**2)

In [18]:
R_expr

Matrix([
[                                                                                                                                                                                                \nabla_{m} \nabla_{n} \tau_{2}/(2*\tau_{2}**2) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}/(4*\tau_{2}**3) - 3*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}/(4*\tau_{2}**3),                                                                                                                                                                          -\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) + \nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}/(2*\tau_{2}**2) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}/(4*\tau_{2}**3) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}/(4*\tau_{2}**2) + 3*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{1}/(4*\tau_{2}**2) - 3*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}*\tau_{1}/(4*\tau_{2}**3)],
[-\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) + \nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}/(2*\tau_{2}**2) 

In [19]:
sol

[((4*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{1}*\tau_{2}**2*c11 + 2*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{2}**2*c01 + 2*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{2}**2*c10 - 2*I*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{2}**2 - 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}**2*\tau_{2}*c11 - 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}*c01 - 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}*c10 + 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{2}**3*c11 - 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{2}**2 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}**2*c11 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}*c01 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}*c10 + 3*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{2}**2*c11 - 2*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{2} - 4*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}*c11 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{2}*c01 - 3*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{2}*c10 + 2*I*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{2} - 

In [6]:
DmPbn = sp.expand(sp.simplify(func2sym(DPb[m,n])))

In [7]:
DmPbn

-I*\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) - \nabla_{m} \nabla_{n} \tau_{2}/(2*\tau_{2}) - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}/(2*\tau_{2}**2) + I*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}/(2*\tau_{2}**2) + I*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{1}/(2*\tau_{2}**2) + \nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}/(2*\tau_{2}**2)

In [ ]:
sp.expand(sp.simplify())

-I*Derivative(tau_1(m, n, r, s), m, n)/(2*tau_2(m, n, r, s)) - Derivative(tau_2(m, n, r, s), m, n)/(2*tau_2(m, n, r, s)) - Derivative(tau_1(m, n, r, s), m)*Derivative(tau_1(m, n, r, s), n)/(2*tau_2(m, n, r, s)**2) + I*Derivative(tau_1(m, n, r, s), m)*Derivative(tau_2(m, n, r, s), n)/(2*tau_2(m, n, r, s)**2) + I*Derivative(tau_1(m, n, r, s), n)*Derivative(tau_2(m, n, r, s), m)/(2*tau_2(m, n, r, s)**2) + Derivative(tau_2(m, n, r, s), m)*Derivative(tau_2(m, n, r, s), n)/(2*tau_2(m, n, r, s)**2)

In [3]:
sp.expand(sp.simplify(func2sym(R_ambn[m,n][0,1])))

-\nabla_{m} \nabla_{n} \tau_{1}/(2*\tau_{2}) + \nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}/(2*\tau_{2}**2) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}/(4*\tau_{2}**3) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}/(4*\tau_{2}**2) + 3*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{1}/(4*\tau_{2}**2) - 3*\nabla_{m} \tau_{2}*\nabla_{n} \tau_{2}*\tau_{1}/(4*\tau_{2}**3)

Matrix([
[-(tau_1(m, n, r, s)**2/(2*tau_2(m, n, r, s)) + tau_2(m, n, r, s)/2)*Derivative(tau_2(m, n, r, s), m)/tau_2(m, n, r, s)**2 - (-tau_1(m, n, r, s)*Derivative(tau_2(m, n, r, s), m)/tau_2(m, n, r, s)**2 + Derivative(tau_1(m, n, r, s), m)/tau_2(m, n, r, s))*tau_1(m, n, r, s)/(2*tau_2(m, n, r, s)), -((2*tau_1(m, n, r, s)*Derivative(tau_1(m, n, r, s), m) + 2*tau_2(m, n, r, s)*Derivative(tau_2(m, n, r, s), m))/tau_2(m, n, r, s) - (tau_1(m, n, r, s)**2 + tau_2(m, n, r, s)**2)*Derivative(tau_2(m, n, r, s), m)/tau_2(m, n, r, s)**2)*tau_1(m, n, r, s)/(2*tau_2(m, n, r, s)) + (tau_1(m, n, r, s)**2/(2*tau_2(m, n, r, s)) + tau_2(m, n, r, s)/2)*(-tau_1(m, n, r, s)*Derivative(tau_2(m, n, r, s), m)/tau_2(m, n, r, s)**2 + Derivative(tau_1(m, n, r, s), m)/tau_2(m, n, r, s))],
[                                                                (-tau_1(m, n, r, s)*Derivative(tau_2(m, n, r, s), m)/tau_2(m, n, r, s)**2 + Derivative(tau_1(m, n, r, s), m)/tau_2(m, n, r, s))/(2*tau_2(m, n, r, s)) + tau_1(m,

In [5]:
fun2sym(R_ambn[m,n])[0,0]

(2*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{1}**3*\tau_{2}**2 + 2*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{1}*\tau_{2}**4 + 2*\nabla_{m} \nabla_{n} \tau_{1}*\tau_{1}*\tau_{2}**2 + 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{1}**2*\tau_{2}**3 + 2*\nabla_{m} \nabla_{n} \tau_{2}*\tau_{2}**5 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}**4*\tau_{2} + 2*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}**2*\tau_{2}**2 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{1}**2*\tau_{2} + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{2}**5 + 2*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\tau_{2}**4 - 2*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{1}**3*\tau_{2}**2 - 4*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{1}**3*\tau_{2} - 2*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}**4 - 4*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}**3 - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{1}*\tau_{2}**2 - 4*\nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\tau_{1}*\tau_{2} + \nabla_{m} \tau_{2}*\nabla_{n}

In [4]:
function_2_symbols(ginv_func)

Matrix([
[\tau_{1}**2/\tau_{2} + \tau_{2}, -\tau_{1}/\tau_{2}],
[             -\tau_{1}/\tau_{2},         1/\tau_{2}]])

In [2]:
DmPbnDrPs = DPb[m,n] * DP[r,s]

In [ ]:
DmPbnDrPs = real_parts(fun2sym(DmPbnDrPs))
DmPbnDrPs = sp.expand(DmPbnDrPs)

In [6]:
DmPbnDrPs

\nabla_{m} \nabla_{n} \tau_{1}*\nabla_{r} \nabla_{s} \tau_{1}/(4*\tau_{2}**2) - \nabla_{m} \nabla_{n} \tau_{1}*\nabla_{r} \tau_{1}*\nabla_{s} \tau_{2}/(4*\tau_{2}**3) - \nabla_{m} \nabla_{n} \tau_{1}*\nabla_{r} \tau_{2}*\nabla_{s} \tau_{1}/(4*\tau_{2}**3) + \nabla_{m} \nabla_{n} \tau_{2}*\nabla_{r} \nabla_{s} \tau_{2}/(4*\tau_{2}**2) + \nabla_{m} \nabla_{n} \tau_{2}*\nabla_{r} \tau_{1}*\nabla_{s} \tau_{1}/(4*\tau_{2}**3) - \nabla_{m} \nabla_{n} \tau_{2}*\nabla_{r} \tau_{2}*\nabla_{s} \tau_{2}/(4*\tau_{2}**3) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\nabla_{r} \nabla_{s} \tau_{2}/(4*\tau_{2}**3) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\nabla_{r} \tau_{1}*\nabla_{s} \tau_{1}/(4*\tau_{2}**4) - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{1}*\nabla_{r} \tau_{2}*\nabla_{s} \tau_{2}/(4*\tau_{2}**4) - \nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\nabla_{r} \nabla_{s} \tau_{1}/(4*\tau_{2}**3) + \nabla_{m} \tau_{1}*\nabla_{n} \tau_{2}*\nabla_{r} \tau_{1}*\nabla_{s} \tau_{2}/(4*\tau_{2}**4) + \nabla_{m} \

In [6]:
print(tau1)

tau_1(m, n, r, s)


In [4]:
SUBS_D0

[(tau_1(m, n, r, s), \tau_{1}),
 (tau_2(m, n, r, s), \tau_{2}),
 (H3(m, n, r, s), H_{3}),
 (F3(m, n, r, s), F_{3})]